In [ ]:
%pip install elasticsearch==8.19.0
%restart_python

### Build awards-v4 with explicit mapping (oxjob #123.2)

One-off notebook to:
1. Fetch awards-v3 mapping (preserve everything else).
2. Override `primary_topic`, `topics`, and `institution_awarded` to `nested` with explicit `keyword` ids — fixes the auto-mapping debt from #123.1 and adds the new field from #123.2 in one shot.
3. Create awards-v4 with the constructed mapping + v3's shard/replica/analyzer settings.
4. Full-sync `openalex.awards.awards_api` → awards-v4 (mirrors `sync_awards.ipynb` pattern).
5. Sanity-check counts and a sample nested query.

After running:
- Update `openalex-elastic-api/settings.py`: `AWARDS_INDEX = "awards-v4"`
- Update `notebooks/elastic/sync_awards.ipynb` CONFIG to `"index_name": "awards-v4"`
- Remove the 8 `.keyword` workarounds from `openalex-elastic-api/awards/fields.py`

In [ ]:
import copy
import json
from pyspark.sql import functions as F
from elasticsearch import Elasticsearch, helpers
import logging

logging.basicConfig(level=logging.WARNING, format='[%(asctime)s]: %(message)s')
log = logging.getLogger(__name__)

ELASTIC_URL = dbutils.secrets.get(scope="elastic", key="elastic_url")

CONFIG = {
    "table_name": "openalex.awards.awards_api",
    "source_index": "awards-v3",
    "target_index": "awards-v4",
}

client = Elasticsearch(
    hosts=[ELASTIC_URL],
    max_retries=3,
    request_timeout=180,
)

# Safety asserts: source must exist, target must not (avoid clobbering a partial v4)
assert client.indices.exists(index=CONFIG['source_index']), \
    f"{CONFIG['source_index']} not found; aborting"
assert not client.indices.exists(index=CONFIG['target_index']), \
    f"{CONFIG['target_index']} already exists; delete it first if rebuilding"

print(f"OK: {CONFIG['source_index']} exists, {CONFIG['target_index']} does not")

In [ ]:
# Fetch v3 mapping verbatim, then surgically override the three problem fields
v3_mapping = client.indices.get_mapping(index=CONFIG['source_index'])[CONFIG['source_index']]['mappings']
v4_mapping = copy.deepcopy(v3_mapping)
v4_mapping.setdefault('properties', {})

# Shared topic sub-structure (used by primary_topic AND each topics[] element)
topic_props = {
    "id":           {"type": "keyword"},
    "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    "score":        {"type": "float"},
    "subfield": {"properties": {
        "id":           {"type": "keyword"},
        "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    }},
    "field": {"properties": {
        "id":           {"type": "keyword"},
        "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    }},
    "domain": {"properties": {
        "id":           {"type": "keyword"},
        "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
    }},
}

# Override topics + primary_topic (do-over of #123.1's auto-mapping)
v4_mapping['properties']['primary_topic'] = {"type": "nested", "properties": topic_props}
v4_mapping['properties']['topics']        = {"type": "nested", "properties": topic_props}

# New field from #123.2 — explicit nested from the start
v4_mapping['properties']['institution_awarded'] = {
    "type": "nested",
    "properties": {
        "id":           {"type": "keyword"},
        "display_name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
        "ror":          {"type": "keyword"},
        "country_code": {"type": "keyword"},
        "type":         {"type": "keyword"},
        "lineage":      {"type": "keyword"},
    },
}

print("v4 mapping properties for our overrides:")
print(json.dumps({k: v4_mapping['properties'][k] for k in ('primary_topic', 'topics', 'institution_awarded')}, indent=2))

In [ ]:
# Copy a defensible subset of v3 index settings (shards, replicas, custom analyzers)
v3_settings_full = client.indices.get_settings(index=CONFIG['source_index'])[CONFIG['source_index']]['settings']['index']

v4_settings = {}
for k in ('number_of_shards', 'number_of_replicas', 'analysis'):
    if k in v3_settings_full:
        v4_settings[k] = v3_settings_full[k]

print(f"v4 settings to apply: {json.dumps(v4_settings, indent=2)}")

client.indices.create(
    index=CONFIG['target_index'],
    body={
        "settings": v4_settings,
        "mappings": v4_mapping,
    },
)
print(f"Created {CONFIG['target_index']}")

In [ ]:
# Full sync from awards_api → awards-v4. Mirrors sync_awards.ipynb pattern.
def send_partition_to_elastic(partition, index_name):
    client = Elasticsearch(
        hosts=[ELASTIC_URL],
        max_retries=3,
        request_timeout=180,
    )

    def generate_actions():
        for row in partition:
            yield {
                "_op_type": "index",
                "_index": index_name,
                "_id": row.id,
                "_source": row._source.asDict(True),
            }

    count = 0
    for success, info in helpers.parallel_bulk(
        client,
        generate_actions(),
        chunk_size=500,
        thread_count=4,
    ):
        count += 1
        if not success:
            print(f"FAILED TO INDEX: {info}")
            raise Exception(f"Failed to index document: {info}")
    print(f"Successfully indexed {count} docs to {index_name}")


df = spark.read.table(CONFIG['table_name'])
df = (
    df
    .withColumn("id", F.concat(F.lit("https://openalex.org/G"), F.col("id")))
    .select("id", F.struct(F.col("*")).alias("_source"))
)
df = df.repartition(96)
print(f"Total rows to sync: {df.count()}")

target = CONFIG['target_index']
df.foreachPartition(lambda p: send_partition_to_elastic(p, target))
print("Sync complete")

In [ ]:
# Refresh and sanity-check
client.indices.refresh(index=CONFIG['target_index'])

v3_count = client.count(index=CONFIG['source_index'])['count']
v4_count = client.count(index=CONFIG['target_index'])['count']
print(f"awards-v3 docs: {v3_count}")
print(f"awards-v4 docs: {v4_count}")
print(f"delta: {v4_count - v3_count}")

# Topic filter on v4 with the NEW mapping (no .keyword suffix needed)
v4_topic = client.search(
    index=CONFIG['target_index'],
    body={
        "size": 0,
        "query": {
            "nested": {
                "path": "primary_topic",
                "query": {"term": {"primary_topic.id": "https://openalex.org/T11668"}}
            }
        },
    },
)
print(f"v4 awards with primary_topic.id = T11668 (nested query, no .keyword): {v4_topic['hits']['total']['value']}")

# Same query on v3 with the .keyword workaround for comparison
v3_topic = client.search(
    index=CONFIG['source_index'],
    body={
        "size": 0,
        "query": {"term": {"primary_topic.id.keyword": "https://openalex.org/T11668"}},
    },
)
print(f"v3 awards with primary_topic.id.keyword = T11668 (existing workaround): {v3_topic['hits']['total']['value']}")
print(f"==> v3 and v4 should agree on this number; if not, mapping needs investigation.")

# New field sanity: institution_awarded nested filter
v4_inst = client.search(
    index=CONFIG['target_index'],
    body={
        "size": 0,
        "query": {
            "nested": {
                "path": "institution_awarded",
                "query": {"term": {"institution_awarded.country_code": "CH"}}
            }
        },
    },
)
print(f"v4 awards with institution_awarded.country_code = CH: {v4_inst['hits']['total']['value']}")

# Compound nested query (the reason we picked nested over object)
v4_compound = client.search(
    index=CONFIG['target_index'],
    body={
        "size": 0,
        "query": {
            "nested": {
                "path": "institution_awarded",
                "query": {"bool": {"must": [
                    {"term": {"institution_awarded.country_code": "DE"}},
                    {"term": {"institution_awarded.type": "education"}},
                ]}}
            }
        },
    },
)
print(f"v4 awards with institution_awarded country=DE AND type=education on SAME element: {v4_compound['hits']['total']['value']}")